# Image Classification and Real-Time Prediction — CIFAR-10

This notebook walks through the full pipeline for the project brief:

1. Data collection & exploration
2. Data preprocessing (augmentation, normalization, splitting)
3. Model development (custom CNN and MobileNetV2 transfer learning)
4. Model training (early stopping, dropout)
5. Model evaluation (accuracy, precision, recall, confusion matrix)
6. Model explainability (Grad-CAM)

The reusable pipeline code lives in `../src/`; this notebook imports and
demonstrates it interactively. For full training runs, prefer running
`python src/train.py` directly (this notebook uses a smaller number of
epochs for fast, interactive iteration).


In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt

from config import CLASS_NAMES, IMG_SIZE, MODEL_TYPE
from data_preprocessing import load_raw_data, explore_dataset, get_datasets
from model import build_model
from gradcam import explain_prediction

%matplotlib inline


ModuleNotFoundError: No module named 'tensorflow'

## 1. Data Collection & Exploration

In [ ]:
(x_train, y_train), (x_test, y_test) = load_raw_data()
explore_dataset(x_train, y_train, x_test, y_test)


In [ ]:
# Visualize a few sample images per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls in range(10):
    idx = np.where(y_train == cls)[0][0]
    ax = axes[cls // 5, cls % 5]
    ax.imshow(x_train[idx])
    ax.set_title(CLASS_NAMES[cls])
    ax.axis("off")
plt.suptitle("One sample per class")
plt.tight_layout()
plt.show()


## 2. Data Preprocessing

`get_datasets()` handles resizing, normalization to `[0, 1]`, on-the-fly
augmentation (random flip, rotation, brightness/contrast jitter, zoom) for
the training set, and a stratified train/validation split.


In [ ]:
train_ds, val_ds, test_ds, (x_test_raw, y_test_raw) = get_datasets()

for imgs, lbls in train_ds.take(1):
    print("Batch shape:", imgs.shape, "| Label shape:", lbls.shape)
    print("Pixel range after preprocessing:", float(imgs.numpy().min()), "-", float(imgs.numpy().max()))


In [ ]:
# Visualize augmentation: same source images, run through the training pipeline twice
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for imgs, lbls in train_ds.take(1):
    for i in range(6):
        axes[0, i].imshow(imgs[i])
        axes[0, i].set_title(CLASS_NAMES[int(lbls[i])])
        axes[0, i].axis("off")
for imgs, lbls in train_ds.take(1):
    for i in range(6):
        axes[1, i].imshow(imgs[i])
        axes[1, i].axis("off")
plt.suptitle("Top: batch 1 (augmented) | Bottom: batch 2 (different augmentation draw)")
plt.tight_layout()
plt.show()


## 3. Model Development

Two selectable architectures (see `src/model.py`):

- **`custom_cnn`**: a CNN built from scratch (Conv/BatchNorm/Dropout blocks + GAP head).
- **`mobilenet`**: transfer learning on top of MobileNetV2 pretrained on ImageNet.

Both compile with Adam, sparse categorical cross-entropy, and accuracy tracking.


In [ ]:
model = build_model(model_type=MODEL_TYPE)
model.summary()


## 4. Model Training

Uses early stopping (on validation loss), learning-rate reduction on
plateau, and checkpointing of the best weights. For a full run, use:

```bash
python ../src/train.py --model_type mobilenet --epochs 30
```

Below is a short interactive run for demonstration.


In [ ]:
import tensorflow as tf

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks, verbose=2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout()
plt.show()


## 5. Model Evaluation

Accuracy, precision, recall, F1, and a confusion matrix. For the full
evaluation script (which also saves a report + heatmap to `outputs/`), run:

```bash
python ../src/evaluate.py
```


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_raw[: len(y_pred)]

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 6. Model Explainability (Bonus): Grad-CAM

Grad-CAM highlights which regions of the input image most influenced the
model's predicted class. `explain_prediction()` (from `src/gradcam.py`)
takes a PIL image and returns the predicted class, full probability
distribution, and a heatmap overlay.


In [ ]:
from PIL import Image

sample_idx = 0
sample_img = Image.fromarray(x_test_raw[sample_idx])

pred_index, probs, overlay = explain_prediction(sample_img, model, MODEL_TYPE, IMG_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_img)
axes[0].set_title(f"Original (true: {CLASS_NAMES[int(y_test_raw[sample_idx])]})")
axes[0].axis("off")
axes[1].imshow(overlay)
axes[1].set_title(f"Grad-CAM (pred: {CLASS_NAMES[pred_index]}, {probs[pred_index]*100:.1f}%)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## Next Steps

- Train longer with the full CIFAR-10 dataset via `python src/train.py --model_type mobilenet --epochs 30`.
- Try fine-tuning: `python src/train.py --model_type mobilenet --fine_tune_at 100`.
- Launch the interactive app: `streamlit run app/streamlit_app.py`.
- Launch the REST API: `python api/flask_app.py` (or via Docker, see `api/Dockerfile`).
